# Blinkit Retail Intelligence Platform: Exploratory Data Analysis

## Objective
Load and inspect the Blinkit Grocery Sales dataset to explore sales trends, product categories, and outlet performance using Python.

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("../data/BlinkIT-Grocery-Data.csv")

# Display the first 5 records
df.head()

,Item Fat Content,Item Identifier,Item Type,Outlet Establishment Year,Outlet Identifier,Outlet Location Type,Outlet Size,Outlet Type,Item Visibility,Item Weight,Sales,Rating
0,Regular,FDX32,Fruits and Vegetables,2012,OUT049,Tier 1,Medium,Supermarket Type1,0.100014,15.10,145.4786,5.0
1,Low Fat,NCB42,Health and Hygiene,2022,OUT018,Tier 3,Medium,Supermarket Type2,0.008596,11.80,115.3492,5.0
2,Regular,FDR28,Frozen Foods,2016,OUT046,Tier 1,Small,Supermarket Type1,0.025896,13.85,165.0210,5.0
3,Regular,FDL50,Canned,2014,OUT013,Tier 3,High,Supermarket Type1,0.042278,12.15,126.5046,5.0
4,Low Fat,DRI25,Soft Drinks,2015,OUT045,Tier 2,Small,Supermarket Type1,0.033970,19.60,55.1614,5.0


## Structural Audit & Missing Value Check

Explore the dataset's dimensions, data types, and count missing entries to ensure our calculations won't break later.

In [2]:
#Print the total number of rows and columns
print("Dataset Shape (Rows, Columns):", df.shape)

#Check data types and look for missing (null) values
df.info()

Dataset Shape (Rows, Columns): (8523, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item Fat Content           8523 non-null   object 
 1   Item Identifier            8523 non-null   object 
 2   Item Type                  8523 non-null   object 
 3   Outlet Establishment Year  8523 non-null   int64  
 4   Outlet Identifier          8523 non-null   object 
 5   Outlet Location Type       8523 non-null   object 
 6   Outlet Size                8523 non-null   object 
 7   Outlet Type                8523 non-null   object 
 8   Item Visibility            8523 non-null   float64
 9   Item Weight                7060 non-null   float64
 10  Sales                      8523 non-null   float64
 11  Rating                     8523 non-null   float64
dtypes: float64(4), int64(1), object(7)
memory usage: 799.2+ KB


## Checking for Text Inconsistencies in Categories

Before building charts, we need to inspect our categorical columns. We will check the unique values inside columns like product fat content and store sizes to ensure there are no spelling errors or duplicate categories holding different names.

In [3]:
# Check unique values in the fat content column to spot spelling inconsistencies
print("Unique entries in Item Fat Content:")
print(df['Item Fat Content'].unique())

# Check unique entries for outlet sizes to see how stores are classified
print("\nUnique entries in Outlet Size:")
print(df['Outlet Size'].unique())

# Check unique entries for the city location tiers
print("\nUnique entries in Outlet Location Type:")
print(df['Outlet Location Type'].unique())

# Verify if the drop-down categories for Item Type have any duplicates or typos
print("\nUnique product types found in inventory:")
print(df['Item Type'].unique())

# Verify if the store classifications for Outlet Type are standardized
print("\nUnique store operational types:")
print(df['Outlet Type'].unique())

Unique entries in Item Fat Content:
['Regular' 'Low Fat' 'low fat' 'LF' 'reg']

Unique entries in Outlet Size:
['Medium' 'Small' 'High']

Unique entries in Outlet Location Type:
['Tier 1' 'Tier 3' 'Tier 2']

Unique product types found in inventory:
['Fruits and Vegetables' 'Health and Hygiene' 'Frozen Foods' 'Canned'
 'Soft Drinks' 'Household' 'Snack Foods' 'Meat' 'Breads' 'Hard Drinks'
 'Others' 'Dairy' 'Breakfast' 'Baking Goods' 'Seafood' 'Starchy Foods']

Unique store operational types:
['Supermarket Type1' 'Supermarket Type2' 'Grocery Store'
 'Supermarket Type3']


## Standardizing Product Fat Content Categories

To ensure our charts and filters group data accurately, we map all historical text typos and abbreviations ('low fat', 'LF', 'reg') into two clean, standardized categories: 'Low Fat' and 'Regular'.

In [4]:
# Define a dictionary to map historical typos to clean categories
fat_content_rules = {
    'low fat': 'Low Fat',
    'LF': 'Low Fat',
    'reg': 'Regular'
} 

# Apply the correction rules to the column
df['Item Fat Content'] = df['Item Fat Content'].replace(fat_content_rules)

# Verify that our categories are now perfectly clean
print("Updated unique entries in Item Fat Content:")
print(df['Item Fat Content'].unique())

Updated unique entries in Item Fat Content:
['Regular' 'Low Fat']


## Handling Missing Values in Item Weight Using Barcode Lookup

Instead of filling missing product weights with a generic store-wide average, we group the data by 'Item Identifier' (the product barcode) to find its true weight from other store records. We use a secondary safety net based on the product category if an item has no historical weight records anywhere.

In [5]:
#Find the average weight for each unique product ID across the whole dataset
item_specific_weight = df.groupby('Item Identifier')['Item Weight'].transform('mean')

#Fill the missing values using those product-specific averages
df['Item Weight'] = df['Item Weight'].fillna(item_specific_weight)

# 3. Safety net: For any items that still have a missing weight, fill them with the average of their product category
category_specific_weight = df.groupby('Item Type')['Item Weight'].transform('mean')
df['Item Weight'] = df['Item Weight'].fillna(category_specific_weight)

# 4. Verify that the missing value count for Item Weight dropped to zero
print("Remaining missing values in Item Weight:")
print(df['Item Weight'].isnull().sum())

Remaining missing values in Item Weight:
0


In [6]:
df.head()

,Item Fat Content,Item Identifier,Item Type,Outlet Establishment Year,Outlet Identifier,Outlet Location Type,Outlet Size,Outlet Type,Item Visibility,Item Weight,Sales,Rating
0,Regular,FDX32,Fruits and Vegetables,2012,OUT049,Tier 1,Medium,Supermarket Type1,0.100014,15.10,145.4786,5.0
1,Low Fat,NCB42,Health and Hygiene,2022,OUT018,Tier 3,Medium,Supermarket Type2,0.008596,11.80,115.3492,5.0
2,Regular,FDR28,Frozen Foods,2016,OUT046,Tier 1,Small,Supermarket Type1,0.025896,13.85,165.0210,5.0
3,Regular,FDL50,Canned,2014,OUT013,Tier 3,High,Supermarket Type1,0.042278,12.15,126.5046,5.0
4,Low Fat,DRI25,Soft Drinks,2015,OUT045,Tier 2,Small,Supermarket Type1,0.033970,19.60,55.1614,5.0


##  Feature Engineering & Baseline Cost Calculations

To power our dynamic profit simulator app, we engineer two business metrics:
1. Estimated Cost: Assuming an industry-standard 30% baseline gross profit margin, the cost to procure the item is 70% of its MRP.
2. Baseline Profit: The net profit generated before any interactive discount simulations are applied.

Finally, we export this clean dataset as a new file to power our Streamlit application.

In [8]:
#Calculate the total wholesale cost of the items sold (70% of total sales)
df['Estimated_Cost'] = df['Sales'] * 0.70

#Calculate the baseline profit made on total sales (30% margin)
df['Baseline_Profit'] = df['Sales'] * 0.30

#Save this beautifully prepared dataframe into our data folder for Streamlit
df.to_csv("../data/blinkit_processed.csv", index=False)

print("Feature engineering complete! 'blinkit_processed.csv' has been successfully created using Sales metrics.")

Feature engineering complete! 'blinkit_processed.csv' has been successfully created using Sales metrics.
